# Design Patterns in Python

This notebook introduces nine classic design patterns, grouped into creational, structural and behavioural patterns. For each one you'll see the problem it solves, a short Python implementation, and where you've probably met it already. We finish with a mini-project that uses four patterns together.

### Learning Objectives

By the end of this notebook you will be able to:

1. Explain what design patterns are and why they matter
2. Implement **creational** patterns: Singleton, Factory, Builder
3. Implement **structural** patterns: Adapter, Decorator, Facade
4. Implement **behavioural** patterns: Observer, Strategy, Command
5. Choose a suitable pattern for a real-world problem
6. Recognise patterns in popular Python libraries

### Prerequisites

- The **OOP Foundations** and **SOLID Principles** notebooks
- Python 3.10 or newer
- Abstract base classes (`from abc import ABC, abstractmethod`) and `@dataclass`

### How to Use This Notebook

- Run the cells **from top to bottom**. Some later cells reuse imports from earlier ones.
- **Try it** cells and **Practice Exercises** are for you. Write your answer first, then open *Show solution* to compare.
- Each section ends with a **Quick recap**. Read only the recaps when revising before an exam.

---

## What Are Design Patterns?

> *"Design patterns are reusable solutions to commonly occurring problems in software design."*

Think of them as **recipes**. They aren't libraries you install or code you copy and paste. They are well-known shapes for your classes that experienced developers have used again and again.

### Why Learn Them?

| Benefit | What it means for you |
|---|---|
| Common vocabulary | Saying "use an Observer here" is quicker than explaining the whole mechanism |
| Proven solutions | You don't have to reinvent the wheel |
| Readable code | Other developers recognise the shape and understand your intent |
| Testable code | Most patterns lead to loosely coupled code that is easy to test |
| Interviews | Design patterns are a common interview topic |

### The Three Families

| Family | Question it answers | Patterns in this notebook |
|---|---|---|
| Creational | How do we **create** objects? | Singleton, Factory, Builder |
| Structural | How do we **combine** objects? | Adapter, Decorator, Facade |
| Behavioural | How do objects **talk** to each other? | Observer, Strategy, Command |

One word of warning before we start: a pattern is a tool, not a goal. Use one when you actually have the problem it solves.

**Quick recap**

- A design pattern is a well-known, reusable shape for solving a common design problem.
- Patterns come in three families: creational, structural and behavioural.
- Only use a pattern when you have the problem it solves.

---

## 1. Creational Patterns

Creational patterns are about **how objects get created**: how many, which class, and in how many steps.

### 1.1 Singleton

**Intent:** make sure a class has only **one** instance, and give everyone access to it.

Real-world examples:

- **Database connection pool**: one pool manages all connections
- **Application configuration**: one config object for the whole app
- **Logger**: one logger so all messages go to the same place

Use it when exactly one object should coordinate something across the whole program.

First, the problem: every call creates a new connection.

In [1]:
# Problem: Multiple database connections waste resources

class DatabaseConnection:
    def __init__(self):
        print("🔌 Creating new database connection...")
        self.connected = True

# Every call creates a NEW connection - wasteful!
conn1 = DatabaseConnection()
conn2 = DatabaseConnection()
conn3 = DatabaseConnection()

print(f"\nAre they the same object? {conn1 is conn2}")  # False - different objects!

🔌 Creating new database connection...
🔌 Creating new database connection...
🔌 Creating new database connection...

Are they the same object? False


Three separate objects, three connections. Now let's make the class hand back the same object every time.

In [2]:
# Solution: Singleton Pattern

class DatabaseConnection:
    _instance = None  # Class-level storage for the single instance
    
    def __new__(cls):
        """Called BEFORE __init__. Controls object creation."""
        if cls._instance is None:
            print("🔌 Creating database connection (first time only!)")
            cls._instance = super().__new__(cls)
            cls._instance.connected = True
        return cls._instance
    
    def query(self, sql):
        return f"Executing: {sql}"

# Test it!
conn1 = DatabaseConnection()
conn2 = DatabaseConnection()
conn3 = DatabaseConnection()

print(f"\nAre they the same object? {conn1 is conn2}")  # True!
print(f"Same IDs? {id(conn1)} == {id(conn2)} == {id(conn3)}")

🔌 Creating database connection (first time only!)

Are they the same object? True
Same IDs? 4491995728 == 4491995728 == 4491995728


What just happened: `__new__` is the method that actually **creates** an object, and it runs before `__init__`. The first time, `_instance` is `None`, so we create the object and store it on the class. Every later call skips creation and returns the stored object.

Two things to watch out for:

- If you also write an `__init__`, it runs on **every** call, even when the old object is returned. That's why the setup (`connected = True`) is inside `__new__` here.
- A singleton is global state. It makes testing harder, so use it sparingly.

#### The Pythonic Singleton: Just Use a Module

In Python the simplest singleton is a **module**. A module is only loaded once, so any object created at module level exists only once.

In [3]:
# config.py (conceptual - as a module)
# This IS a singleton - Python modules are only loaded once!

class _Config:
    def __init__(self):
        self.debug = True
        self.database_url = "localhost:5432"
        self.api_key = "secret_key_123"

# The single instance
config = _Config()

# Now anyone can: from config import config
print(f"Debug mode: {config.debug}")
print(f"Database: {config.database_url}")

Debug mode: True
Database: localhost:5432


If this code lived in `config.py`, every file that does `from config import config` would get the very same object. No `__new__` tricks needed.

#### Try It: Singleton Logger

Create a `Logger` class where:

- only **one** instance can exist
- `log(message)` prints the message with a timestamp
- `message_count` tracks how many messages have been logged in total

In [4]:
# Try it: your code here
from datetime import datetime


class Logger:
    _instance = None

    def __new__(cls):
        # TODO: create the instance only once, and start message_count at 0
        pass

    def log(self, message):
        # TODO: print with a timestamp and count the message
        pass

# Test your code (uncomment to run):
# logger1 = Logger()
# logger2 = Logger()
# logger1.log("First message")
# logger2.log("Second message")
# logger1.log("Third message")
# assert logger1 is logger2           # same instance
# assert logger1.message_count == 3   # three messages, from either name
# print("Try it passed!")

<details markdown="1">
<summary>Show solution</summary>

```python
from datetime import datetime


class Logger:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.message_count = 0
        return cls._instance

    def log(self, message):
        print(f"[{datetime.now()}] {message}")
        self.message_count += 1
```

</details>

Notice the count is 3, not 2: `logger1` and `logger2` are the same object, so all three messages go to one counter.

### 1.2 Factory

**Intent:** create objects without the caller needing to know the exact class. The caller asks for "email", and the factory decides which class to build.

Real-world examples:

- **Game characters**: create a warrior, mage or archer based on the player's choice
- **Document parsers**: create a PDF, Word or JSON parser based on the file type
- **Payment processors**: create a Stripe, PayPal or eSewa processor

Use it when you don't know in advance which class you'll need, or when you want creation logic in one place.

In [5]:
# Simple Factory Example: Notification System

from abc import ABC, abstractmethod

# Abstract base class
class Notification(ABC):
    @abstractmethod
    def send(self, message: str) -> str:
        pass

# Concrete implementations
class EmailNotification(Notification):
    def send(self, message: str) -> str:
        return f"📧 Email sent: {message}"

class SMSNotification(Notification):
    def send(self, message: str) -> str:
        return f"📱 SMS sent: {message}"

class PushNotification(Notification):
    def send(self, message: str) -> str:
        return f"🔔 Push notification: {message}"

# The Factory!
class NotificationFactory:
    @staticmethod
    def create(notification_type: str) -> Notification:
        """Factory method - creates the right notification type."""
        factories = {
            "email": EmailNotification,
            "sms": SMSNotification,
            "push": PushNotification
        }
        
        if notification_type not in factories:
            raise ValueError(f"Unknown notification type: {notification_type}")
        
        return factories[notification_type]()

# Usage - the client doesn't need to know concrete classes!
notification = NotificationFactory.create("email")
print(notification.send("Hello, World!"))

notification = NotificationFactory.create("sms")
print(notification.send("Your code is 1234"))

notification = NotificationFactory.create("push")
print(notification.send("New message received!"))

📧 Email sent: Hello, World!
📱 SMS sent: Your code is 1234
🔔 Push notification: New message received!


What just happened: the calling code only ever deals with the `Notification` interface and a string like `"sms"`. It never imports `SMSNotification` directly. To add WhatsApp notifications, we'd write one new class and add one line to the dictionary. Asking for an unknown type raises a clear `ValueError`.

### 1.3 Builder

**Intent:** build a complex object **step by step**, instead of in one giant constructor call.

Real-world examples:

- **Meal builder**: a burger with optional toppings
- **Query builder**: an SQL query built piece by piece
- **Email builder**: an email with optional CC, BCC and attachments

Use it when an object has many optional parts, to avoid the *telescoping constructor* problem (an `__init__` with a long list of parameters). Here's that problem:

In [6]:
# Problem: Too many constructor parameters!

class Pizza:
    def __init__(self, size, cheese=True, pepperoni=False, mushrooms=False,
                 onions=False, bacon=False, olives=False, extra_cheese=False,
                 stuffed_crust=False, thin_crust=False):
        self.size = size
        self.cheese = cheese
        self.pepperoni = pepperoni
        self.mushrooms = mushrooms
        self.onions = onions
        self.bacon = bacon
        self.olives = olives
        self.extra_cheese = extra_cheese
        self.stuffed_crust = stuffed_crust
        self.thin_crust = thin_crust

# Hard to read and error-prone!
pizza = Pizza("large", True, True, False, True, True, False, True, False, False)
# What does the 5th True mean?

Nobody reading `Pizza("large", True, True, False, True, ...)` can tell which `True` means what. Now the builder version:

In [7]:
# Solution: Builder Pattern

class Pizza:
    def __init__(self):
        self.size = "medium"
        self.toppings = []
        self.crust = "regular"
    
    def __str__(self):
        toppings_str = ", ".join(self.toppings) if self.toppings else "no toppings"
        return f"🍕 {self.size.title()} pizza with {self.crust} crust and {toppings_str}"


class PizzaBuilder:
    """Builds a pizza step by step with a fluent interface."""
    
    def __init__(self):
        self.pizza = Pizza()
    
    def set_size(self, size: str) -> 'PizzaBuilder':
        self.pizza.size = size
        return self  # Return self for chaining!
    
    def add_pepperoni(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🔴 pepperoni")
        return self
    
    def add_mushrooms(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🍄 mushrooms")
        return self
    
    def add_onions(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🧅 onions")
        return self
    
    def add_bacon(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🥓 bacon")
        return self
    
    def add_extra_cheese(self) -> 'PizzaBuilder':
        self.pizza.toppings.append("🧀 extra cheese")
        return self
    
    def set_thin_crust(self) -> 'PizzaBuilder':
        self.pizza.crust = "thin"
        return self
    
    def set_stuffed_crust(self) -> 'PizzaBuilder':
        self.pizza.crust = "stuffed"
        return self
    
    def build(self) -> Pizza:
        """Return the final pizza."""
        return self.pizza


# Beautiful, readable pizza creation!
my_pizza = (PizzaBuilder()
    .set_size("large")
    .add_pepperoni()
    .add_mushrooms()
    .add_extra_cheese()
    .set_stuffed_crust()
    .build())

print(my_pizza)

🍕 Large pizza with stuffed crust and 🔴 pepperoni, 🍄 mushrooms, 🧀 extra cheese


What just happened: every builder method changes the pizza and then `return self`. Because each call returns the builder, we can chain the next call straight onto it. `build()` hands back the finished `Pizza`.

To be fair to Python: keyword arguments with defaults (`Pizza(size="large", pepperoni=True)`) already fix a lot of the readability problem. A builder really pays off when the steps have logic, ordering rules or validation.

**Quick recap**

- **Singleton**: one shared instance. In Python, a module-level object is often enough.
- **Factory**: the caller asks for a kind of object by name; the factory picks the class.
- **Builder**: build a complex object step by step; methods `return self` so calls can be chained.

---

## 2. Structural Patterns

Structural patterns are about **how objects are combined** to form bigger structures.

### 2.1 Adapter

**Intent:** make a class with the "wrong" interface fit the interface your code expects.

Real-world analogies:

- **Power adapter**: a US plug in a Nepali socket
- **Audio adapter**: a 3.5mm headphone jack into a USB-C port

Use it when you have to use an existing class (often a third-party or legacy one) whose methods don't match what you need, and you can't change it.

In [8]:
# Adapter Pattern: Integrating a legacy payment system

from abc import ABC, abstractmethod

# Old legacy system (can't modify this!)
class LegacyPaymentSystem:
    """Old payment system with different method names."""
    def make_payment(self, amount_cents: int, card_number: str) -> bool:
        print(f"💳 Legacy: Processing {amount_cents} cents from card {card_number[-4:]}")
        return True

# New interface that our app expects
class PaymentProcessor(ABC):
    @abstractmethod
    def process(self, amount: float, payment_info: dict) -> bool:
        pass

# The Adapter - bridges old and new!
class LegacyPaymentAdapter(PaymentProcessor):
    """Adapts the legacy system to our new interface."""
    
    def __init__(self, legacy_system: LegacyPaymentSystem):
        self.legacy = legacy_system
    
    def process(self, amount: float, payment_info: dict) -> bool:
        # Convert dollars to cents
        amount_cents = int(amount * 100)
        # Extract card number from payment_info
        card_number = payment_info.get("card_number", "")
        # Call the legacy system
        return self.legacy.make_payment(amount_cents, card_number)

# Usage
legacy = LegacyPaymentSystem()
adapter = LegacyPaymentAdapter(legacy)

# Now we can use the new interface!
result = adapter.process(
    amount=99.99,
    payment_info={"card_number": "4111111111111234", "cvv": "123"}
)
print(f"Payment successful: {result}")

💳 Legacy: Processing 9999 cents from card 1234
Payment successful: True


What just happened: our app wants `process(amount_in_dollars, payment_info)`, but the old system only offers `make_payment(amount_in_cents, card_number)`. The adapter sits in between. It converts dollars to cents, pulls out the card number, and calls the old method. Neither our app nor the legacy class had to change.

### 2.2 Decorator

**Intent:** add extra behaviour to an object by **wrapping** it, instead of creating a subclass for every combination.

> **Note:** this is related to, but not the same as, Python's `@decorator` syntax. Both wrap something to add behaviour; the `@` syntax wraps **functions**, while this pattern wraps **objects**.

Real-world examples:

- **Coffee shop**: base coffee + milk + sugar + whipped cream
- **Game power-ups**: character + speed boost + shield
- **Text formatting**: text + bold + italic + underline

In [9]:
# Decorator Pattern: Coffee Shop

from abc import ABC, abstractmethod

# Base component
class Coffee(ABC):
    @abstractmethod
    def get_description(self) -> str:
        pass
    
    @abstractmethod
    def get_cost(self) -> float:
        pass

# Concrete component
class Espresso(Coffee):
    def get_description(self) -> str:
        return "☕ Espresso"
    
    def get_cost(self) -> float:
        return 2.00

# Base decorator
class CoffeeDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self._coffee = coffee

# Concrete decorators
class Milk(CoffeeDecorator):
    def get_description(self) -> str:
        return f"{self._coffee.get_description()} + 🥛 Milk"
    
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.50

class Sugar(CoffeeDecorator):
    def get_description(self) -> str:
        return f"{self._coffee.get_description()} + 🍬 Sugar"
    
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.25

class WhippedCream(CoffeeDecorator):
    def get_description(self) -> str:
        return f"{self._coffee.get_description()} + 🍨 Whipped Cream"
    
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.75

# Build a fancy coffee!
my_coffee = Espresso()  # Start with base
my_coffee = Milk(my_coffee)  # Add milk
my_coffee = Sugar(my_coffee)  # Add sugar
my_coffee = WhippedCream(my_coffee)  # Add whipped cream

print(my_coffee.get_description())
print(f"Total: ${my_coffee.get_cost():.2f}")

☕ Espresso + 🥛 Milk + 🍬 Sugar + 🍨 Whipped Cream
Total: $3.50


What just happened: each add-on holds the coffee it wraps (`self._coffee`) and adds its own part on top. When we ask the outermost `WhippedCream` for its cost, it asks `Sugar`, which asks `Milk`, which asks `Espresso`. That's 2.00 + 0.50 + 0.25 + 0.75 = **3.50**.

Without this pattern we'd need a subclass for every combination (`EspressoWithMilk`, `EspressoWithMilkAndSugar`, ...).

### 2.3 Facade

**Intent:** give a simple front door to a complicated set of classes.

Real-world examples:

- **Home theatre**: one "watch movie" button controls the TV, speakers, lights and player
- **Bank withdrawal**: one simple call hides account checks, balance checks and ledger updates
- **Computer start-up**: one power button starts the CPU, memory, disks and so on

In [10]:
# Facade Pattern: Home Theater System

# Complex subsystems
class TV:
    def on(self): print("📺 TV is ON")
    def off(self): print("📺 TV is OFF")
    def set_input(self, source): print(f"📺 Input set to {source}")

class SoundSystem:
    def on(self): print("🔊 Sound system is ON")
    def off(self): print("🔊 Sound system is OFF")
    def set_volume(self, level): print(f"🔊 Volume set to {level}")
    def set_surround_mode(self): print("🔊 Surround sound activated")

class StreamingPlayer:
    def on(self): print("▶️ Streaming player is ON")
    def off(self): print("▶️ Streaming player is OFF")
    def play(self, movie): print(f"▶️ Playing: {movie}")
    def pause(self): print("▶️ Paused")

class Lights:
    def dim(self, level): print(f"💡 Lights dimmed to {level}%")
    def on(self): print("💡 Lights ON at 100%")

# The Facade - simple interface!
class HomeTheaterFacade:
    def __init__(self):
        self.tv = TV()
        self.sound = SoundSystem()
        self.player = StreamingPlayer()
        self.lights = Lights()
    
    def watch_movie(self, movie: str):
        """One simple method to start movie night!"""
        print("\n🎬 Starting Movie Night...\n")
        self.lights.dim(10)
        self.tv.on()
        self.tv.set_input("HDMI1")
        self.sound.on()
        self.sound.set_surround_mode()
        self.sound.set_volume(65)
        self.player.on()
        self.player.play(movie)
        print("\n🍿 Enjoy your movie!\n")
    
    def end_movie(self):
        """Shut everything down."""
        print("\n🛑 Ending Movie Night...\n")
        self.player.off()
        self.sound.off()
        self.tv.off()
        self.lights.on()

# Simple usage!
theater = HomeTheaterFacade()
theater.watch_movie("The Matrix")
theater.end_movie()


🎬 Starting Movie Night...

💡 Lights dimmed to 10%
📺 TV is ON
📺 Input set to HDMI1
🔊 Sound system is ON
🔊 Surround sound activated
🔊 Volume set to 65
▶️ Streaming player is ON
▶️ Playing: The Matrix

🍿 Enjoy your movie!


🛑 Ending Movie Night...

▶️ Streaming player is OFF
🔊 Sound system is OFF
📺 TV is OFF
💡 Lights ON at 100%


The person using `HomeTheaterFacade` only needs to know two methods. The subsystems are still there if someone needs fine control, but everyday use is one line.

**Quick recap**

- **Adapter**: converts one interface into another so existing classes can work together.
- **Decorator**: wraps an object to add behaviour; wrappers can be stacked.
- **Facade**: one simple interface in front of many complicated parts.

---

## 3. Behavioural Patterns

Behavioural patterns are about **how objects communicate** and share responsibility.

### 3.1 Observer

**Intent:** when one object changes, all the objects that are interested get told automatically.

You'll also hear it called **publish-subscribe (pub/sub)**, an **event system**, or the **listener pattern**.

Real-world examples:

- **Newsletter**: subscribers get every new article
- **Stock prices**: several displays update when a price changes
- **Social media**: followers are notified of new posts

In [11]:
# Observer Pattern: Weather Station

from abc import ABC, abstractmethod
from typing import List

# Observer interface
class WeatherObserver(ABC):
    @abstractmethod
    def update(self, temperature: float, humidity: float):
        pass

# Subject (Observable)
class WeatherStation:
    def __init__(self):
        self._observers: List[WeatherObserver] = []
        self._temperature = 0.0
        self._humidity = 0.0
    
    def subscribe(self, observer: WeatherObserver):
        self._observers.append(observer)
        print(f"✅ {observer.__class__.__name__} subscribed")
    
    def unsubscribe(self, observer: WeatherObserver):
        self._observers.remove(observer)
        print(f"❌ {observer.__class__.__name__} unsubscribed")
    
    def notify_observers(self):
        for observer in self._observers:
            observer.update(self._temperature, self._humidity)
    
    def set_measurements(self, temperature: float, humidity: float):
        print(f"\n🌡️ Weather Station: New data - {temperature}°C, {humidity}% humidity")
        self._temperature = temperature
        self._humidity = humidity
        self.notify_observers()  # Automatically notify all observers!

# Concrete Observers
class PhoneDisplay(WeatherObserver):
    def update(self, temperature: float, humidity: float):
        print(f"   📱 Phone: It's {temperature}°C outside")

class WindowDisplay(WeatherObserver):
    def update(self, temperature: float, humidity: float):
        print(f"   🖼️ Window Display: {temperature}°C | {humidity}% humidity")

class AlertSystem(WeatherObserver):
    def update(self, temperature: float, humidity: float):
        if temperature > 35:
            print(f"   🚨 ALERT: Extreme heat warning! {temperature}°C")
        elif temperature < 0:
            print(f"   🚨 ALERT: Freezing warning! {temperature}°C")
        else:
            print(f"   ✅ Alert System: Temperature normal")

# Demo
station = WeatherStation()

# Create and subscribe observers
phone = PhoneDisplay()
window = WindowDisplay()
alert = AlertSystem()

station.subscribe(phone)
station.subscribe(window)
station.subscribe(alert)

# Update weather - all observers get notified!
station.set_measurements(22.5, 65)
station.set_measurements(38.0, 40)

# Unsubscribe one display: it stops receiving updates
station.unsubscribe(window)
station.set_measurements(-2.0, 80)

✅ PhoneDisplay subscribed
✅ WindowDisplay subscribed
✅ AlertSystem subscribed

🌡️ Weather Station: New data - 22.5°C, 65% humidity
   📱 Phone: It's 22.5°C outside
   🖼️ Window Display: 22.5°C | 65% humidity
   ✅ Alert System: Temperature normal

🌡️ Weather Station: New data - 38.0°C, 40% humidity
   📱 Phone: It's 38.0°C outside
   🖼️ Window Display: 38.0°C | 40% humidity
   🚨 ALERT: Extreme heat warning! 38.0°C
❌ WindowDisplay unsubscribed

🌡️ Weather Station: New data - -2.0°C, 80% humidity
   📱 Phone: It's -2.0°C outside
   🚨 ALERT: Freezing warning! -2.0°C


What just happened: the `WeatherStation` (the *subject*) keeps a list of observers. Whenever new data arrives, it loops over the list and calls `update` on each one. The station doesn't know or care what each observer does with the data. After `window` unsubscribes, it no longer gets updates.

Adding a new display means writing one new observer class. The station doesn't change, which is the Open/Closed principle again.

### 3.2 Strategy

**Intent:** define a family of interchangeable algorithms and let the caller pick one at run time.

Real-world examples:

- **Payment methods**: pay by card, cash or mobile wallet
- **Navigation**: fastest, shortest or scenic route
- **Shipping**: express, standard or store pickup
- **Login**: password, OAuth or two-factor

In [12]:
# Strategy Pattern: Shipping Cost Calculator

from abc import ABC, abstractmethod
from dataclasses import dataclass

# Strategy interface
class ShippingStrategy(ABC):
    @abstractmethod
    def calculate(self, weight: float) -> float:
        """Calculate shipping cost based on weight in kg."""
        pass
    
    @property
    @abstractmethod
    def name(self) -> str:
        pass

# Concrete strategies
class StandardShipping(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return weight * 2.50  # $2.50 per kg
    
    @property
    def name(self) -> str:
        return "📦 Standard (5-7 days)"

class ExpressShipping(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return weight * 5.00 + 10.00  # $5 per kg + $10 handling
    
    @property
    def name(self) -> str:
        return "🚀 Express (1-2 days)"

class PickupStrategy(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return 0.00  # Free!
    
    @property
    def name(self) -> str:
        return "🏪 Store Pickup (Free)"

# Context - uses a strategy
@dataclass
class Order:
    items: list
    weight: float
    shipping_strategy: ShippingStrategy = None
    
    def set_shipping(self, strategy: ShippingStrategy):
        self.shipping_strategy = strategy
    
    def get_shipping_cost(self) -> float:
        if not self.shipping_strategy:
            raise ValueError("Please select a shipping method!")
        return self.shipping_strategy.calculate(self.weight)
    
    def __str__(self):
        shipping_cost = self.get_shipping_cost()
        return (f"Order: {self.items}\n"
                f"Weight: {self.weight}kg\n"
                f"Method: {self.shipping_strategy.name}\n"
                f"Shipping: ${shipping_cost:.2f}")

# Usage
order = Order(items=["Laptop", "Mouse"], weight=3.5)

# Try different strategies
print("=" * 40)
order.set_shipping(StandardShipping())
print(order)

print("\n" + "=" * 40)
order.set_shipping(ExpressShipping())
print(order)

print("\n" + "=" * 40)
order.set_shipping(PickupStrategy())
print(order)

Order: ['Laptop', 'Mouse']
Weight: 3.5kg
Method: 📦 Standard (5-7 days)
Shipping: $8.75

Order: ['Laptop', 'Mouse']
Weight: 3.5kg
Method: 🚀 Express (1-2 days)
Shipping: $27.50

Order: ['Laptop', 'Mouse']
Weight: 3.5kg
Method: 🏪 Store Pickup (Free)
Shipping: $0.00


What just happened: the `Order` doesn't contain any shipping maths. It just holds a strategy object and calls `calculate(weight)` on it. Changing the strategy changes the cost:

- Standard: 3.5 × 2.50 = **8.75**
- Express: 3.5 × 5.00 + 10 = **27.50**
- Pickup: **0.00**

You saw this exact shape in the SOLID notebook's discount example.

### 3.3 Command

**Intent:** wrap an action in an object, so it can be stored, queued, logged or **undone**.

Real-world examples:

- **Undo/redo** in text and image editors
- **Game replays**: record and replay player moves
- **Task queues**: schedule work to run later
- **GUI buttons**: each button holds the action it performs

In [13]:
# Command Pattern: Text Editor with Undo

from abc import ABC, abstractmethod
from typing import List

# Command interface
class Command(ABC):
    @abstractmethod
    def execute(self) -> None:
        pass
    
    @abstractmethod
    def undo(self) -> None:
        pass

# Receiver - the actual text document
class TextDocument:
    def __init__(self):
        self.content = ""
    
    def write(self, text: str):
        self.content += text
    
    def erase(self, num_chars: int):
        if num_chars <= 0:
            return  # nothing to erase (note: content[:-0] would wipe everything!)
        self.content = self.content[:-num_chars]
    
    def __str__(self):
        return f"📄 Document: '{self.content}'"

# Concrete Commands
class WriteCommand(Command):
    def __init__(self, document: TextDocument, text: str):
        self.document = document
        self.text = text
    
    def execute(self):
        self.document.write(self.text)
        print(f"✍️ Wrote: '{self.text}'")
    
    def undo(self):
        self.document.erase(len(self.text))
        print(f"↩️ Undid write of: '{self.text}'")

# Invoker - manages command execution and history
class TextEditor:
    def __init__(self, document: TextDocument):
        self.document = document
        self.history: List[Command] = []
    
    def execute(self, command: Command):
        command.execute()
        self.history.append(command)
    
    def undo(self):
        if self.history:
            command = self.history.pop()
            command.undo()
        else:
            print("⚠️ Nothing to undo!")

# Demo
doc = TextDocument()
editor = TextEditor(doc)

editor.execute(WriteCommand(doc, "Hello "))
print(doc)

editor.execute(WriteCommand(doc, "World!"))
print(doc)

editor.execute(WriteCommand(doc, " How are you?"))
print(doc)

print("\n--- Undo operations ---")
editor.undo()
print(doc)

editor.undo()
print(doc)

✍️ Wrote: 'Hello '
📄 Document: 'Hello '
✍️ Wrote: 'World!'
📄 Document: 'Hello World!'
✍️ Wrote: ' How are you?'
📄 Document: 'Hello World! How are you?'

--- Undo operations ---
↩️ Undid write of: ' How are you?'
📄 Document: 'Hello World!'
↩️ Undid write of: 'World!'
📄 Document: 'Hello '


What just happened: each `WriteCommand` remembers what it wrote, so it knows how to undo itself. The `TextEditor` keeps a history list. Undo just pops the last command and calls its `undo()` method.

Because commands are objects, adding redo is simply a second list: push undone commands onto it and re-run them. That's one of the homework problems.

**Quick recap**

- **Observer**: a subject keeps a list of observers and notifies them all when something changes.
- **Strategy**: put each algorithm in its own class and swap them at run time.
- **Command**: wrap an action in an object so it can be stored, queued and undone.

---

## 4. Design Patterns in the Wild

You're already using design patterns without knowing it:

| Pattern | Where you've seen it |
|---|---|
| Singleton | `logging.getLogger("app")` always returns the same logger for a name; `None` is the only `NoneType` object |
| Factory | `datetime.strptime()`, `json.loads()` (you get the right kind of object back without calling its class) |
| Builder (method chaining) | `pandas` method chains like `df.query(...).groupby(...).agg(...)` and SQLAlchemy's `select(...).where(...).order_by(...)` |
| Iterator | Every `for x in collection:` loop |
| Decorator | `@property`, `@staticmethod`, `@functools.lru_cache` (Python's `@` syntax, closely related to the pattern) |
| Observer | Django signals, GUI button click handlers |
| Strategy | The `key=` argument of `sorted()` and `list.sort()` |

Here's the Strategy pattern hiding inside `sorted()`:

In [14]:
# Strategy pattern in action - sorted() with different strategies!

students = [
    {"name": "Alice", "gpa": 3.8, "age": 20},
    {"name": "Bob", "gpa": 3.2, "age": 22},
    {"name": "Charlie", "gpa": 3.9, "age": 19},
]

# Strategy 1: Sort by name
by_name = sorted(students, key=lambda s: s["name"])
print("By name:", [s["name"] for s in by_name])

# Strategy 2: Sort by GPA (descending)
by_gpa = sorted(students, key=lambda s: s["gpa"], reverse=True)
print("By GPA:", [f"{s['name']}: {s['gpa']}" for s in by_gpa])

# Strategy 3: Sort by age
by_age = sorted(students, key=lambda s: s["age"])
print("By age:", [f"{s['name']}: {s['age']}" for s in by_age])

By name: ['Alice', 'Bob', 'Charlie']
By GPA: ['Charlie: 3.9', 'Alice: 3.8', 'Bob: 3.2']
By age: ['Charlie: 19', 'Alice: 20', 'Bob: 22']


`sorted()` is the context; each `key` function is a strategy. Same sorting code, three different behaviours, chosen by the caller.

**Quick recap**

- Python's standard library and popular frameworks use these patterns everywhere.
- Passing a function (like `key=`) is often the lightest way to do Strategy in Python.

---

## 5. Mini-Project: Task Management System

Let's use **four patterns together** in a small task tracker:

1. **Singleton**: `TaskManager` (only one instance)
2. **Factory**: create different task types (Bug, Feature, Improvement)
3. **Observer**: notify team members when a task's status changes
4. **Strategy**: swap how task priority is calculated

Read the code section by section; each block is labelled with its pattern.

In [15]:
# Mini-Project Solution

from abc import ABC, abstractmethod
from datetime import datetime
from typing import List
from dataclasses import dataclass, field

# ============ Observer Pattern ============
class TaskObserver(ABC):
    @abstractmethod
    def on_task_update(self, task: 'Task', old_status: str, new_status: str):
        pass

class TeamMember(TaskObserver):
    def __init__(self, name: str):
        self.name = name
    
    def on_task_update(self, task: 'Task', old_status: str, new_status: str):
        print(f"   📧 {self.name} notified: '{task.title}' changed from {old_status} → {new_status}")

# ============ Strategy Pattern ============
class PriorityStrategy(ABC):
    @abstractmethod
    def calculate(self, task: 'Task') -> int:
        pass

class AgePriorityStrategy(PriorityStrategy):
    """Older tasks get higher priority."""
    def calculate(self, task: 'Task') -> int:
        age_days = (datetime.now() - task.created).days
        return min(age_days + task.base_priority, 10)

class TypePriorityStrategy(PriorityStrategy):
    """Bugs are always higher priority."""
    def calculate(self, task: 'Task') -> int:
        if task.task_type == "Bug":
            return min(task.base_priority + 3, 10)
        return task.base_priority

# ============ Factory Pattern ============
@dataclass
class Task:
    title: str
    task_type: str
    base_priority: int = 5
    status: str = "Open"
    created: datetime = field(default_factory=datetime.now)
    observers: List[TaskObserver] = field(default_factory=list)
    
    def add_observer(self, observer: TaskObserver):
        self.observers.append(observer)
    
    def set_status(self, new_status: str):
        old_status = self.status
        self.status = new_status
        for observer in self.observers:
            observer.on_task_update(self, old_status, new_status)

class TaskFactory:
    @staticmethod
    def create(task_type: str, title: str) -> Task:
        priorities = {"Bug": 8, "Feature": 5, "Improvement": 3}
        return Task(
            title=title,
            task_type=task_type,
            base_priority=priorities.get(task_type, 5)
        )

# ============ Singleton Pattern ============
class TaskManager:
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.tasks = []
            cls._instance.priority_strategy = TypePriorityStrategy()
        return cls._instance
    
    def add_task(self, task: Task):
        self.tasks.append(task)
        print(f"✅ Added: [{task.task_type}] {task.title}")
    
    def set_priority_strategy(self, strategy: PriorityStrategy):
        self.priority_strategy = strategy
    
    def get_sorted_tasks(self) -> List[Task]:
        return sorted(
            self.tasks,
            key=lambda t: self.priority_strategy.calculate(t),
            reverse=True
        )

# ============ Demo ============
print("=" * 50)
print("🏗️ TASK MANAGEMENT SYSTEM")
print("=" * 50)

# Singleton - get the manager
manager = TaskManager()

# Create team members (observers)
alice = TeamMember("Alice")
bob = TeamMember("Bob")

# Factory - create tasks
bug = TaskFactory.create("Bug", "Login button not working")
feature = TaskFactory.create("Feature", "Add dark mode")
improvement = TaskFactory.create("Improvement", "Optimize database queries")

# Add observers
bug.add_observer(alice)
bug.add_observer(bob)
feature.add_observer(alice)

# Add tasks to manager
manager.add_task(bug)
manager.add_task(feature)
manager.add_task(improvement)

# Observer - update task status
print("\n📢 Status Updates:")
bug.set_status("In Progress")
bug.set_status("Resolved")

# Strategy - see task priorities
print("\n📊 Tasks by Priority (Type Strategy):")
for task in manager.get_sorted_tasks():
    priority = manager.priority_strategy.calculate(task)
    print(f"   [{priority}] [{task.task_type}] {task.title}")

🏗️ TASK MANAGEMENT SYSTEM
✅ Added: [Bug] Login button not working
✅ Added: [Feature] Add dark mode
✅ Added: [Improvement] Optimize database queries

📢 Status Updates:
   📧 Alice notified: 'Login button not working' changed from Open → In Progress
   📧 Bob notified: 'Login button not working' changed from Open → In Progress
   📧 Alice notified: 'Login button not working' changed from In Progress → Resolved
   📧 Bob notified: 'Login button not working' changed from In Progress → Resolved

📊 Tasks by Priority (Type Strategy):
   [10] [Bug] Login button not working
   [5] [Feature] Add dark mode
   [3] [Improvement] Optimize database queries


What just happened:

- `TaskFactory.create` chose a starting priority from the task type (Bug 8, Feature 5, Improvement 3).
- Alice and Bob were observers on the bug, so both were notified on each status change. Only Alice watches the feature.
- `TypePriorityStrategy` adds 3 to bugs, so the bug comes first with priority 10 (capped at 10).

Now swap the strategy and confirm the manager really is a singleton:

In [16]:
# Strategy: switch to age-based priority at run time
manager.set_priority_strategy(AgePriorityStrategy())

print("Tasks by Priority (Age Strategy):")
for task in manager.get_sorted_tasks():
    priority = manager.priority_strategy.calculate(task)
    print(f"   [{priority}] [{task.task_type}] {task.title}")

# Singleton: asking again gives the same manager
print("\nSame manager?", TaskManager() is manager)

Tasks by Priority (Age Strategy):
   [8] [Bug] Login button not working
   [5] [Feature] Add dark mode
   [3] [Improvement] Optimize database queries

Same manager? True


All tasks were created a moment ago, so their age is 0 days and the age strategy just shows the base priorities. If these tasks were a week old, each would gain 7 points (up to the cap of 10). The manager's code didn't change at all; only the strategy object did.

**Quick recap**

- Real systems combine several patterns; each one solves a different part of the problem.
- Factory centralises creation, Observer spreads updates, Strategy keeps rules swappable, Singleton gives one shared manager.

---

## 6. Practice Exercises

Five exercises, one pattern each. Write your code in the cell, uncomment the tests, and run it. Only then open the solution.

### Exercise 1: Singleton Cache

Create a `Cache` singleton that stores key-value pairs with an optional TTL (time-to-live, in seconds). `get` should return `None` for a missing or expired key.

In [17]:
# Exercise 1: Singleton Cache
import time


class Cache:
    _instance = None

    def __new__(cls):
        pass  # TODO: implement singleton

    def set(self, key, value, ttl=None):
        pass  # TODO: store value with optional TTL

    def get(self, key):
        pass  # TODO: return value, or None if missing/expired

# Test your code (uncomment to run):
# cache1 = Cache()
# cache2 = Cache()
# cache1.set("user", "Alice")
# assert cache2.get("user") == "Alice"
# assert cache1 is cache2
# cache1.set("token", "abc", ttl=0.1)
# assert cache2.get("token") == "abc"
# time.sleep(0.2)
# assert cache2.get("token") is None
# assert cache2.get("missing") is None
# print("Exercise 1 passed!")

<details markdown="1">
<summary>Show solution</summary>

```python
import time


class Cache:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._data = {}
        return cls._instance

    def set(self, key, value, ttl=None):
        expires_at = time.time() + ttl if ttl is not None else None
        self._data[key] = (value, expires_at)

    def get(self, key):
        if key not in self._data:
            return None
        value, expires_at = self._data[key]
        if expires_at is not None and time.time() > expires_at:
            del self._data[key]
            return None
        return value
```

</details>

### Exercise 2: Shape Factory

Create a factory that produces `Circle`, `Rectangle` and `Triangle` objects, each with `area()` and `perimeter()`. `ShapeFactory.create("circle", radius=5)` should work, and an unknown shape should raise `ValueError`.

Hint for the triangle (sides `a`, `b`, `c`): with `s = (a + b + c) / 2`, the area is `sqrt(s(s-a)(s-b)(s-c))` (Heron's formula).

In [18]:
# Exercise 2: Shape Factory
from abc import ABC, abstractmethod
import math


class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass

    @abstractmethod
    def perimeter(self) -> float:
        pass


class Circle(Shape):
    pass  # TODO


class Rectangle(Shape):
    pass  # TODO


class Triangle(Shape):
    pass  # TODO


class ShapeFactory:
    @staticmethod
    def create(shape_type: str, **kwargs) -> Shape:
        pass  # TODO

# Test your code (uncomment to run):
# circle = ShapeFactory.create("circle", radius=5)
# assert round(circle.area(), 2) == 78.54
# rect = ShapeFactory.create("rectangle", width=4, height=6)
# assert rect.perimeter() == 20
# tri = ShapeFactory.create("triangle", a=3, b=4, c=5)
# assert tri.area() == 6 and tri.perimeter() == 12
# try:
#     ShapeFactory.create("hexagon")
#     assert False, "should have raised ValueError"
# except ValueError:
#     pass
# print("Exercise 2 passed!")

<details markdown="1">
<summary>Show solution</summary>

```python
from abc import ABC, abstractmethod
import math


class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass

    @abstractmethod
    def perimeter(self) -> float:
        pass


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


class Triangle(Shape):
    def __init__(self, a: float, b: float, c: float):
        self.a, self.b, self.c = a, b, c

    def area(self) -> float:
        s = self.perimeter() / 2
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

    def perimeter(self) -> float:
        return self.a + self.b + self.c


class ShapeFactory:
    _shapes = {"circle": Circle, "rectangle": Rectangle, "triangle": Triangle}

    @staticmethod
    def create(shape_type: str, **kwargs) -> Shape:
        if shape_type not in ShapeFactory._shapes:
            raise ValueError(f"Unknown shape: {shape_type}")
        return ShapeFactory._shapes[shape_type](**kwargs)
```

</details>

### Exercise 3: Recipe Builder

Create a `RecipeBuilder` so that this works:

```python
recipe = (RecipeBuilder()
    .set_name("Pancakes")
    .add_ingredient("Flour", "2 cups")
    .add_ingredient("Milk", "1 cup")
    .add_step("Mix dry ingredients")
    .set_cooking_time(15)
    .build())
```

The finished `Recipe` should have `name`, `ingredients` (a list of `(item, amount)` tuples), `steps` (a list of strings) and `cooking_time` (minutes).

In [19]:
# Exercise 3: Recipe Builder
class Recipe:
    def __init__(self):
        self.name = ""
        self.ingredients = []
        self.steps = []
        self.cooking_time = 0


class RecipeBuilder:
    # TODO: set_name, add_ingredient, add_step, set_cooking_time, build
    pass

# Test your code (uncomment to run):
# recipe = (RecipeBuilder()
#     .set_name("Pancakes")
#     .add_ingredient("Flour", "2 cups")
#     .add_ingredient("Milk", "1 cup")
#     .add_step("Mix dry ingredients")
#     .set_cooking_time(15)
#     .build())
# assert recipe.name == "Pancakes"
# assert recipe.ingredients == [("Flour", "2 cups"), ("Milk", "1 cup")]
# assert recipe.steps == ["Mix dry ingredients"]
# assert recipe.cooking_time == 15
# print("Exercise 3 passed!")

<details markdown="1">
<summary>Show solution</summary>

```python
class Recipe:
    def __init__(self):
        self.name = ""
        self.ingredients = []
        self.steps = []
        self.cooking_time = 0


class RecipeBuilder:
    def __init__(self):
        self.recipe = Recipe()

    def set_name(self, name: str) -> "RecipeBuilder":
        self.recipe.name = name
        return self

    def add_ingredient(self, item: str, amount: str) -> "RecipeBuilder":
        self.recipe.ingredients.append((item, amount))
        return self

    def add_step(self, step: str) -> "RecipeBuilder":
        self.recipe.steps.append(step)
        return self

    def set_cooking_time(self, minutes: int) -> "RecipeBuilder":
        self.recipe.cooking_time = minutes
        return self

    def build(self) -> Recipe:
        return self.recipe
```

</details>

### Exercise 4: Event System (Observer)

Create an `EventEmitter` class with:

- `on(event_name, callback)`: subscribe a function to an event
- `emit(event_name, *args)`: call every subscribed function with `*args`
- `off(event_name, callback)`: unsubscribe

Emitting an event nobody listens to should do nothing (no error).

In [20]:
# Exercise 4: Event System
class EventEmitter:
    # TODO: on, emit, off
    pass

# Test your code (uncomment to run):
# received = []
# def greet(name):
#     received.append(f"Hello {name}")
# def shout(name):
#     received.append(name.upper())
#
# events = EventEmitter()
# events.on("login", greet)
# events.on("login", shout)
# events.emit("login", "Asha")
# assert received == ["Hello Asha", "ASHA"]
# events.off("login", shout)
# events.emit("login", "Ram")
# assert received == ["Hello Asha", "ASHA", "Hello Ram"]
# events.emit("logout")  # nobody listening: no error
# print("Exercise 4 passed!")

<details markdown="1">
<summary>Show solution</summary>

```python
class EventEmitter:
    def __init__(self):
        self._listeners = {}

    def on(self, event_name, callback):
        self._listeners.setdefault(event_name, []).append(callback)

    def emit(self, event_name, *args):
        for callback in self._listeners.get(event_name, []):
            callback(*args)

    def off(self, event_name, callback):
        if callback in self._listeners.get(event_name, []):
            self._listeners[event_name].remove(callback)
```

</details>

### Exercise 5: Sorting Strategies

Write three strategy classes, each with a `sort(students)` method that returns a **new** sorted list:

- `ByName`: alphabetically by name
- `ByGPA`: highest GPA first
- `ByCredits`: most credits first

Then write `StudentList(students, strategy)` with a `set_strategy(strategy)` method and a `sorted_names()` method that returns the names in the current strategy's order.

In [21]:
# Exercise 5: Sorting Strategies
students = [
    {"name": "Charlie", "gpa": 3.9, "credits": 90},
    {"name": "Alice", "gpa": 3.5, "credits": 120},
    {"name": "Bob", "gpa": 3.7, "credits": 60},
]


class ByName:
    pass  # TODO


class ByGPA:
    pass  # TODO


class ByCredits:
    pass  # TODO


class StudentList:
    pass  # TODO

# Test your code (uncomment to run):
# roster = StudentList(students, ByName())
# assert roster.sorted_names() == ["Alice", "Bob", "Charlie"]
# roster.set_strategy(ByGPA())
# assert roster.sorted_names() == ["Charlie", "Bob", "Alice"]
# roster.set_strategy(ByCredits())
# assert roster.sorted_names() == ["Alice", "Charlie", "Bob"]
# assert students[0]["name"] == "Charlie"  # original list unchanged
# print("Exercise 5 passed!")

<details markdown="1">
<summary>Show solution</summary>

```python
class ByName:
    def sort(self, students):
        return sorted(students, key=lambda s: s["name"])


class ByGPA:
    def sort(self, students):
        return sorted(students, key=lambda s: s["gpa"], reverse=True)


class ByCredits:
    def sort(self, students):
        return sorted(students, key=lambda s: s["credits"], reverse=True)


class StudentList:
    def __init__(self, students, strategy):
        self.students = students
        self.strategy = strategy

    def set_strategy(self, strategy):
        self.strategy = strategy

    def sorted_names(self):
        return [s["name"] for s in self.strategy.sort(self.students)]
```

</details>

## Homework

These are bigger, open-ended problems to do at home. There's no single right answer, so no solutions are given here. Bring your design to class.

### Problem 1: Plugin System (Factory + Strategy)

Design a plugin system where:

- a `PluginFactory` creates plugins by name
- each plugin implements a `run()` method
- the main app can load and run plugins dynamically

### Problem 2: Undo/Redo File Editor (Command)

Extend the Command pattern example to support:

- **redo**
- more command types, such as `DeleteCommand` and `ReplaceCommand`
- saving the command history to a log file

### Problem 3: Stock Market Simulator (Observer + Strategy)

Create a stock market simulator where:

- `Stock` objects notify investors when the price changes
- investors use different trading strategies: `BuyLowSellHigh`, `DayTrader`, `LongTermHolder`
- each strategy decides whether to buy or sell based on price movements

### Problem 4: Report Generator (Builder)

Create a report builder that can produce different formats:

```python
report = (ReportBuilder()
    .set_title("Q4 Sales Report")
    .add_section("Summary", summary_data)
    .add_chart("sales_chart.png")
    .add_table(sales_table)
    .set_footer("Confidential")
    .build(format="pdf"))  # or "html", "markdown"
```

### Problem 5: Design Pattern Analysis (Written)

Pick a Python library you use regularly (for example `requests`, `pandas`, Django or Flask) and find **at least 3 design patterns** in its API. For each one, explain:

- which pattern it is
- where in the API you see it
- why it's a good choice there

---

## Summary & Cheat Sheet

| Family | Pattern | Purpose | Python hint |
|---|---|---|---|
| Creational | Singleton | Only one instance | Override `__new__`, or just use a module-level object |
| | Factory | Create objects without naming the exact class | A dictionary mapping names to classes |
| | Builder | Build complex objects step by step | Methods `return self` for chaining |
| Structural | Adapter | Make incompatible interfaces work together | Wrap the old object, translate the calls |
| | Decorator | Add behaviour by wrapping objects | Wrapper holds the wrapped object and calls it |
| | Facade | Simple front door to a complex system | One class with a few high-level methods |
| Behavioural | Observer | Notify many objects when one changes | A list of subscribers, loop and call `update` |
| | Strategy | Swap algorithms at run time | A strategy object, or simply a function (`key=`) |
| | Command | Wrap actions as objects for undo/queueing | `execute()` and `undo()` plus a history list |

### What's Next

The **Event-Driven Programming** notebook builds on the Observer pattern: instead of a weather station calling its displays, you'll see whole programs organised around events and handlers.